In [4]:
import scanpy as sc
import pandas as pd
import numpy as np

# Load the Stage 18 
practicedata = sc.read_h5ad("Stage18_Data.h5ad")

# Inspect the dataset dimensions (cells x genes)
print(practicedata)

# check the data
print(practicedata.obs.tail())

AnnData object with n_obs × n_vars = 30508 × 22806
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.3.9', 'seurat_clusters'
    var: 'vst.mean', 'vst.variance', 'vst.variance.expected', 'vst.variance.standardized', 'vst.variable'
    uns: 'neighbors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances'
    layers: None (.X)
                                 orig.ident  nCount_RNA  nFeature_RNA  \
pax6_Mutant_TTTGTTGCACGAGAAC-1  pax6_Mutant      3870.0          2131   
pax6_Mutant_TTTGTTGCAGCTGAGA-1  pax6_Mutant      4277.0          1975   
pax6_Mutant_TTTGTTGCATGCTGCG-1  pax6_Mutant      5171.0          2716   
pax6_Mutant_TTTGTTGTCCTTCGAC-1  pax6_Mutant     10081.0          3138   
pax6_Mutant_TTTGTTGTCTCTGCTG-1  pax6_Mutant      7673.0          3571   

                                RNA_snn_res.3.9  seurat_clusters  
pax6_Mutant_TTTGTTGCACGAGAAC-1               38               38  
pax6_Mutant_TTTGTTGCAGCTGAGA-1               56               56  
pax6

/var/folders/k3/vg_c2k5x4mbcx6ktwfvjsddw0000gn/T/ipykernel_90153/1567270659.py:6: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  practicedata = sc.read_h5ad("Stage18_Data.h5ad")


In [5]:
# check the string and what data is prvides 
print(practicedata.obs['orig.ident'].unique())

<ArrowStringArray>
['six3_WT', 'six3_Mutant', 'pax6_WT', 'pax6_Mutant']
Length: 4, dtype: str


In [6]:
#seperate wt and mut to infer the gene regualotry networks
adata_pax6_wt = practicedata[practicedata.obs['orig.ident'] == 'pax6_WT'].copy()
adata_pax6_mut = practicedata[practicedata.obs['orig.ident'] == 'pax6_Mutant'].copy()

print("Pax6 Wild-Type object shape:", adata_pax6_wt.shape)
print("Pax6 Mutant object shape:", adata_pax6_mut.shape)

Pax6 Wild-Type object shape: (6399, 22806)
Pax6 Mutant object shape: (8582, 22806)


In [7]:
import loompy as lp
import numpy as np

In [8]:
cluster_counts = adata_pax6_wt.obs['seurat_clusters'].value_counts()
print(cluster_counts)

seurat_clusters
10    282
2     245
7     245
9     234
0     229
28    224
8     218
5     202
31    201
1     196
4     173
11    166
14    146
15    143
12    136
41    135
6     128
24    127
19    125
20    122
48    122
34    117
17    113
29    112
13    112
27    108
21    107
25    106
18    102
26     98
30     98
37     91
32     87
22     86
23     86
35     85
16     80
33     80
36     77
44     74
40     73
43     65
45     64
38     64
42     59
39     54
47     49
53     47
49     42
58     42
55     40
52     38
54     38
51     36
56     29
57     28
59      9
50      3
46      1
Name: count, dtype: int64


In [9]:
# i decding if i should focus on the cluster 6 and 9 which is the retina and my program wont tun but i am afraid that is going to run ebcuase some ifdea is competence

In [10]:
full_cluster_counts = practicedata.obs['seurat_clusters'].value_counts()
print(full_cluster_counts)

seurat_clusters
0     1115
1     1035
2     1024
3     1022
4     1017
5      975
6      906
7      903
8      855
9      749
10     725
11     710
12     684
13     680
14     662
15     652
16     649
17     647
18     638
19     614
20     586
21     582
22     573
23     567
24     560
25     556
26     546
27     504
28     497
29     486
30     476
31     468
32     450
33     444
34     432
35     417
36     372
37     362
38     349
39     346
40     334
41     328
42     324
43     319
44     318
45     307
46     298
47     289
48     278
49     262
50     258
51     240
52     228
53     192
54     175
55     173
56     123
57     101
58      79
59      47
Name: count, dtype: int64


In [11]:
import pandas as pd
from arboreto.algo import grnboost2
from arboreto.utils import load_tf_names

In [12]:

adata_pax6_wt.obs.head()

,orig.ident,nCount_RNA,nFeature_RNA,RNA_snn_res.3.9,seurat_clusters
pax6_WT_AAACCCAAGACCATTC-1,pax6_WT,11622.0,3653,16,16
pax6_WT_AAACCCAAGAGGATGA-1,pax6_WT,11989.0,3820,32,32
pax6_WT_AAACCCAAGTAAACGT-1,pax6_WT,3535.0,2083,37,37
pax6_WT_AAACCCAGTCAAAGCG-1,pax6_WT,5397.0,2747,9,9
pax6_WT_AAACCCAGTTGGCTAT-1,pax6_WT,3945.0,2066,32,32


In [13]:
print (adata_pax6_wt.shape)

(6399, 22806)


In [14]:
print(adata_pax6_wt.var_names[:300])

Index(['LOC101732307', 'pi16', 'dok1', 'LOC101730410', 'mrps26', 'm1ap',
       'loxl3', 'bbc3', 'htra2', 'LOC116408404',
       ...
       'ablim2', 'man2b2', 'ppp2r2c', 'wfs1', 'jakmip1', 'LOC116408438',
       'crmp1', 'evc', 'evc2', 'stk32b'],
      dtype='str', length=300)


In [15]:
import pandas as pd
from arboreto.algo import grnboost2
from arboreto.utils import load_tf_names

In [16]:
pip install setuptools


Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
import numpy as np
import os, glob
import pickle

from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2


In [18]:
import pandas as pd
import numpy as np
from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2

In [19]:
ex_matrix = pd.DataFrame(
    adata_pax6_wt.X.toarray() if hasattr(adata_pax6_wt.X, "toarray") else adata_pax6_wt.X,
    index=adata_pax6_wt.obs_names,
    columns=adata_pax6_wt.var_names
)
print("pax6_wt expression matrix shape:", ex_matrix.shape)

pax6_wt expression matrix shape: (6399, 22806)


In [20]:
tf_filename = "pax6_wt_tfs.txt"
with open(tf_filename, "w") as f:
    for gene in adata_pax6_wt.var_names:
        f.write(f"{gene}\n")

print(f"Transcription factor file '{tf_filename}' created successfully with {len(adata_pax6_wt.var_names)} candidate genes.")

Transcription factor file 'pax6_wt_tfs.txt' created successfully with 22806 candidate genes.


In [21]:
from arboreto.algo import grnboost2
from arboreto.utils import load_tf_names

Loaded 22806 transcription factors for pax6_wt.


/Users/marinmarcillac/Desktop/DS_PAX6/PAX6_GRN/venv/lib/python3.13/site-packages/distributed/node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 60622 instead
  warnings.warn(


Dask client initialized: <Client: 'tcp://127.0.0.1:60623' processes=5 threads=10, memory=16.00 GiB>
Running GRNBoost2 for pax6_wt...
preparing dask client
parsing input
creating dask graph
not shutting down client, client was created externally
finished


TypeError: Must supply at least one delayed object